# Training pipeline
Setup training variables in `training/config.yaml`. Download and store the transaction table and it's metadata under `base_data_dir` as defined in the config.

In [7]:
from pathlib import Path
import os

# Make sure we are in the root directory
def set_project_root(marker="pyproject.toml"):
    path = Path.cwd()
    for parent in [path, *path.parents]:
        if (parent / marker).exists():
            os.chdir(parent)
            return parent
    raise FileNotFoundError(f"Could not find {marker} in any parent directory")
set_project_root()

PosixPath('/Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp')

In [8]:
# Source: https://github.com/VectorInstitute/midst-toolkit/blob/main/examples/training/single_table/run_training.py
from hydra import initialize, compose
from omegaconf import OmegaConf

import pickle
from logging import INFO
from midst_toolkit.common.config import ClavaDDPMDiffusionConfig
from midst_toolkit.common.logger import TOOLKIT_LOGGER, log
from midst_toolkit.common.variables import DEVICE
from midst_toolkit.models.clavaddpm.data_loaders import load_tables
from midst_toolkit.models.clavaddpm.train import ClavaDDPMModelArtifacts, clava_training


# Preventing some excessive logging
TOOLKIT_LOGGER.setLevel(INFO)

In [12]:
ROOT = Path.cwd()
REFERENEC_ROOT = ROOT / "implementations" / "tabular_data" / "single_table" 
# Set data and output directories
base_data_dir = REFERENEC_ROOT / "data"
base_output_dir = REFERENEC_ROOT / "results"

## Load and initialize hydra config

In [11]:
# Context manager ensures global state is cleaned up after initialization
with initialize(version_base=None, config_path="."):
    # Load config.yaml and pass optional command-line style overrides
    cfg = compose(config_name="config")

# View the configuration as a standard YAML string
print(OmegaConf.to_yaml(cfg))


diffusion_config:
  d_layers:
  - 512
  - 512
  dropout: 0.0
  num_timesteps: 2
  model_type: mlp
  iterations: 2
  batch_size: 4
  lr: 0.0006
  gaussian_loss_type: mse
  weight_decay: 1.0e-05
  scheduler: cosine
  data_split_ratios:
  - 0.99
  - 0.005
  - 0.005



## Load the Table
IMPORTANT: code expects `{table}_domain.json` and `dataset_meta.json` files under `base_data_dir`

In [13]:
log(INFO, f"Loading data from {base_data_dir}...")
tables, relation_order, _ = load_tables(Path(base_data_dir))
relation_order

INFO :      Loading data from /Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/tabular_data/single_table/data...
INFO :      Training data ratio is 1, so the data will not be split into training and test sets.
INFO :      Train dataframe shape: (16000, 8)
INFO :      Total dataframe shape: (16000, 8)
INFO :      Numerical data shape: (16000, 4)
INFO :      Categorical data shape: (16000, 4)


[(None, 'trans')]

## Train the model

In [14]:
log(INFO, "Training model...")
diffusion_config = ClavaDDPMDiffusionConfig(**cfg.diffusion_config)
tables, _ = clava_training(
    tables,
    relation_order,
    Path(base_output_dir),
    diffusion_config,
    device=DEVICE,
)
log(INFO, "Model trained successfully.")


INFO :      Training model...
INFO :      Training None -> trans model from scratch
INFO :      No cache_dir provided. Will not attempt to load or save transformed dataset from/to cache
INFO :      No NaN processing policy specified.
INFO :      Model params: ModelParameters(diffusion_parameters=DiffusionParameters(layers_dimensions=[512, 512], dropout=0.0, input_dimension=0, output_dimension=0, embedding_dimension=0, n_blocks=0, block_dimension=0, hidden_dimension=0, dropout_first=0, dropout_second=0), input_dimension=np.int64(8), num_classes=0, is_target_conditioned=<IsTargetConditioned.NONE: 'none'>)
INFO :      Getting model: mlp
INFO :      Saving None -> trans model to /Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/tabular_data/single_table/results/models/None_trans_ckpt.pkl
INFO :      Model trained successfully.


## Save Results

In [15]:

results_file = Path(base_output_dir) / "models" / "None_trans_ckpt.pkl"
log(INFO, f"Checking the results from {results_file}...")

with open(results_file, "rb") as f:
    result = pickle.load(f)

# Asserting the results are the correct type
assert isinstance(result, ClavaDDPMModelArtifacts)

log(INFO, f"Result size (in bytes): {results_file.stat().st_size}")


INFO :      Checking the results from /Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/tabular_data/single_table/results/models/None_trans_ckpt.pkl...
INFO :      Result size (in bytes): 3122328
